In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

In [11]:
assumptions = Assumptions()

#TODO load the fuselage here
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Defining the standard aircraft with the standard planform
To be used when ppl don't wanna build their own planform

In [26]:
standard_wing = Planform(aspect_ratio=27, span=2.667, sweep_quarter_deg=15., taper=.5, thickness_to_chord=0.12, cm_quarter_chord=0,
                         wetted_surface_ratio=1.07, interference_factor=1.0, clmax=1.25, flap=False)

In [27]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

standard_wing.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
standard_wing.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
standard_wing.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
standard_wing.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [28]:
standard_wing.mass_cache = 3.
standard_wing.x_cg_cache = 0.2

# We consider the thing to be tailed

In [29]:
tail = TailFinder(fixed, 14., .7, .8).find_planforms(standard_wing)

print(tail[0].wing_area / standard_wing.wing_area)

0.3654935220424802


In [16]:
for t in tail:
    t.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    t.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    t.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    t.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [8]:
ac = Aircraft(fixed, [standard_wing] + tail)


# Requirement check for the aircraft

In [9]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [10]:
failed_reqs = list()
for requirement, label in zip(requirements, requirement_labels):
    if not requirement.assess(ac):
        failed_reqs.append(label)

if len(failed_reqs):
    print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
    print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
    print(f"Failed: {failed_reqs}")
    print()

Fuel available: 10.0 kg
Fuel required: 5.359012886653261 kg
Difference: 4.640987113346739 kg
theta fails
ac mass: 28.8, 0.7113878660017187
MainWing: AR=23, tc=0.12, sweep=14.999999999999998 deg, cmac=0
Failed: ['Landing Gear']



In [16]:
#TODO: ctrl surface sizing